In [1]:
# Run this single cell in Google Colab or Kaggle (with T4 GPU enabled)
!pip install -q git+https://github.com/huggingface/transformers.git accelerate fastapi uvicorn pillow numpy
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import base64, io, subprocess, threading, time, re
import numpy as np
import torch
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from transformers import Sam3Processor, Sam3Model

MODEL_ID = "unsloth/sam3.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

print(f"Loading {MODEL_ID} on {device} ({dtype})...")
processor = Sam3Processor.from_pretrained(MODEL_ID)
model = Sam3Model.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device).eval()
print("SAM 3.1 loaded Ready!")

app = FastAPI()

class SegmentRequest(BaseModel):
    image_b64: str
    prompts: list[str]
    threshold: float = 0.35

def _mask_iou_bool(a: np.ndarray, b: np.ndarray) -> float:
    inter = int(np.logical_and(a, b).sum())
    if inter == 0:
        return 0.0
    union = int(np.logical_or(a, b).sum())
    return float(inter) / max(float(union), 1.0)

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "device": device}

@app.post("/segment")
def segment(req: SegmentRequest):
    img_bytes = base64.b64decode(req.image_b64)
    image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    w, h = image.size
    max_area = int(w * h * 0.25)
    min_area = 250
    detections = []
    accepted_masks = []

    for prompt in req.prompts:
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(device, dtype)
        with torch.inference_mode():
            outputs = model(**inputs)
        results = processor.post_process_instance_segmentation(
            outputs,
            threshold=req.threshold,
            mask_threshold=0.5,
            target_sizes=inputs.get("original_sizes").tolist(),
        )[0]

        if len(results["masks"]) > 0:
            scores = results["scores"].cpu().numpy()
            order = np.argsort(-scores)
            for idx in order:
                idx = int(idx)
                mask_bool = results["masks"][idx].cpu().numpy().astype(bool)
                area = int(mask_bool.sum())
                if area < min_area or area > max_area:
                    continue
                if any(_mask_iou_bool(mask_bool, prev) > 0.45 for prev in accepted_masks):
                    continue
                accepted_masks.append(mask_bool)
                mask_np = mask_bool.astype(np.uint8) * 255
                box = [int(x) for x in results["boxes"][idx].cpu().tolist()]
                score = float(scores[idx])
                buf = io.BytesIO()
                Image.fromarray(mask_np).save(buf, format="PNG")
                mask_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
                detections.append({
                    "label": prompt,
                    "score": score,
                    "bbox_2d": box,
                    "mask_png_b64": mask_b64,
                })
                break

    return {"model": MODEL_ID, "detections": detections}

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8005, log_level="warning"), daemon=True).start()
time.sleep(2)

proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8005"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("\n========================================================")
        print(f"✅ PASTE THIS INTO YOUR LOCAL .env AS SAM31_ENDPOINT_URL:")
        print(f"SAM31_ENDPOINT_URL={match.group(0)}")
        print("========================================================\n")
        break

Installing dependencies...
Loading unsloth/sam3.1 on cuda (torch.bfloat16)...
SAM 3.1 loaded Ready!

✅ PASTE THIS INTO YOUR LOCAL .env AS SAM31_ENDPOINT_URL:
SAM31_ENDPOINT_URL=https://example-tunnel.trycloudflare.com

